# 83 - Métricas de Tracking (MOTA, MOTP) — Base

In [ ]:
import numpy as np
def iou(a,b):
    xA=max(a[0],b[0]); yA=max(a[1],b[1]); xB=min(a[2],b[2]); yB=min(a[3],b[3])
    inter=max(0,xB-xA)*max(0,yB-yA)
    ua=(a[2]-a[0])*(a[3]-a[1]); ub=(b[2]-b[0])*(b[3]-b[1])
    union=ua+ub-inter
    return inter/union if union>0 else 0.0
def mota_motp(gt,pred,thr=0.5):
    FP=FN=IDSW=0; sum_iou=0; matches=0; N_gt=0; prev={}
    for t in sorted(gt.keys()|pred.keys()):
        g=gt.get(t,[]); p=pred.get(t,[]); N_gt+=len(g)
        usedp=set(); usedg=set(); cur={}
        for gi,gb in enumerate(g):
            best=-1; bests=0
            for pj,pb in enumerate(p):
                if pj in usedp: continue
                s=iou(gb,pb)
                if s>bests: bests=s; best=pj
            if bests>=thr:
                usedp.add(best); usedg.add(gi); cur[best]=gi; sum_iou+=bests; matches+=1
                if best in prev and prev[best]!=gi: IDSW+=1
        FP += len(p)-len(usedp); FN += len(g)-len(usedg); prev=cur
    MOTA=1-(FP+FN+IDSW)/max(N_gt,1); MOTP=(sum_iou/max(matches,1))
    return MOTA, MOTP, dict(FP=FP,FN=FN,IDSW=IDSW,N_gt=N_gt,matches=matches)
gt={0:[[10,10,30,30],[60,60,80,80]],1:[[12,10,32,30],[58,60,78,80]]}
pr={0:[[11,11,31,31],[61,60,81,80]],1:[[70,70,90,90],[13,10,33,30]]}
MOTA,MOTP,bd=mota_motp(gt,pr,0.5); print('MOTA',round(MOTA,3),'MOTP',round(MOTP,3),bd)
